# Session 12: Building and Training CNN Models with PyTorch

**Learning goals:**

- Describe the main stages of a CNN classifier.
- Build a simple CNN using PyTorch layers and a custom `forward()` method.
- Explain and implement the PyTorch training loop.
- Evaluate a trained classifier using held-out predictions and learning curves.
- Freeze a pretrained CNN, replace its classification head and fine-tune its final block.

**Prerequisites:**

- Sessions 9–11, basic Python classes and classification concepts.
- Python with PyTorch, torchvision, pandas, NumPy, Matplotlib and scikit-learn installed.
- Run this notebook from the course folder, using `sample_data/mnist_train_small.csv`.
- Internet access for the first download of pretrained ResNet18 weights.
- A GPU is optional; all examples can run on CPU, although transfer learning is slower.

## Outline

1. Building a CNN Feature Block with Conv2d and MaxPool2d
2. Build a Simple CNN with PyTorch
3. Train a CNN with PyTorch
4. Evaluate the Model
5. Fine-tune a CNN Model: Transfer Learning

## 1. Building a CNN Feature Block with Conv2d and MaxPool2d

Work through [Session 12.1: Building a CNN Feature Block with Conv2d and MaxPool2d](<Session12.1 CNN Feature Blocks and Training Data Preparation.ipynb>) to explore convolution, activation and pooling, then return here to train the same model.

The model uses **Conv2d → ReLU → MaxPool2d → Flatten → Linear**. One 28 × 28 greyscale image becomes 16 feature maps of 14 × 14. Flatten joins these into 3,136 values, and Linear produces ten raw class scores (logits).

| Stage | PyTorch layer | Output shape for a batch of N images |
|---|---|---|
| Input | Image tensor | `[N, 1, 28, 28]` |
| Feature extraction | `nn.Conv2d(1, 16, 3, stride=1, padding=1)` | `[N, 16, 28, 28]` |
| Activation | `nn.ReLU()` | `[N, 16, 28, 28]` |
| Spatial reduction | `nn.MaxPool2d(2, stride=2)` | `[N, 16, 14, 14]` |
| Flatten | `nn.Flatten(start_dim=1)` | `[N, 3136]` |
| Classification | `nn.Linear(3136, 10)` | `[N, 10]` |

The final layer returns logits directly. `nn.CrossEntropyLoss` includes the required log-softmax calculation. Softmax is useful for displaying probabilities and preserves the class selected by `argmax`.


## 2. Build a Simple CNN with PyTorch

This example uses the same `DigitCNN` class and `model` variable as Session 12.1 and its practice notebook.

### 2.1 PyTorch classes, functions and methods

`torch` provides tensor operations, while `torch.nn` provides trainable layers and loss functions. Most `nn` layers are classes: calling a class creates a layer object, and calling that object applies its operation to a tensor.

| API | Role in this model | Key arguments or behaviour |
|---|---|---|
| `nn.Module` | Base class for the complete model | Registers layers assigned as attributes |
| `super().__init__()` | Initialises the base module | Call before assigning layers |
| `nn.Sequential(...)` | Groups layers into an ordered pipeline | Passes each layer's output to the next |
| `nn.Conv2d(...)` | Creates a trainable convolution layer | `in_channels`, `out_channels`, `kernel_size`, `padding` |
| `nn.ReLU()` | Applies the activation operation | Replaces negative values with zero |
| `nn.MaxPool2d(...)` | Reduces spatial size | `kernel_size=2` uses a 2 × 2 pooling window |
| `nn.Flatten(...)` | Makes one feature vector per image | `start_dim=1` preserves the batch dimension |
| `nn.Linear(...)` | Creates a fully connected layer | `in_features` and `out_features` specify its input and output sizes |
| `forward(self, images)` | Defines the model's computation | Connects the feature extractor to the classifier |
| `model(images)` | Runs the model on a batch | Invokes the module's forward machinery; prefer this to calling `forward()` directly |

[PyTorch neural-network modules](https://docs.pytorch.org/docs/stable/nn.html)

In [ ]:
import torch
from torch import nn

class DigitCNN(nn.Module):
    """Extract image features and produce ten digit-class scores."""

    def __init__(self):
        super().__init__()  # Initialise the parent module before adding layers.
        self.features = nn.Sequential(
            nn.Conv2d(
                in_channels=1,  # Accept one greyscale channel.
                out_channels=16,  # Produce 16 feature maps.
                kernel_size=3,  # Apply a 3 × 3 filter.
                stride=1,  # Move one pixel at each step.
                padding=1,  # Keep the 28 × 28 spatial size.
            ),
            nn.ReLU(),  # Replace negative responses with zero.
            nn.MaxPool2d(kernel_size=2, stride=2),  # Reduce each map to 14 × 14.
        )
        self.classifier = nn.Sequential(
            nn.Flatten(start_dim=1),  # Flatten the maps while keeping the batch axis.
            nn.Linear(in_features=16 * 14 * 14, out_features=10),  # Produce ten class scores directly from the flattened features.
        )

    def forward(self, images):
        features = self.features(images)  # Extract and pool local image features.
        return self.classifier(features)  # Convert the features into ten class scores.

torch.manual_seed(42)  # Reproduce the initial random model weights.
model = DigitCNN()  # Create the same feature block and classifier as the lesson.
print(model)
print('Parameters:', sum(parameter.numel() for parameter in model.parameters()))
print('Weights are randomly initialised; this model has not been trained.')


### 2.2 Check the model before training

A temporary batch checks that the layers connect correctly. It does not train the model. Two input images should produce a tensor with shape `[2, 10]`.

`model.parameters()` provides the tensors that an optimiser can update. `numel()` counts the values in each parameter tensor.

In [ ]:
CHECK_BATCH_SIZE = 2
IMAGE_SIZE = 28
NUM_CLASSES = 10
example_images = torch.zeros(CHECK_BATCH_SIZE, 1, IMAGE_SIZE, IMAGE_SIZE)
example_logits = model(example_images)  # Run the forward pass through all layers.
parameter_count = sum(parameter.numel() for parameter in model.parameters())
assert example_logits.shape == (CHECK_BATCH_SIZE, NUM_CLASSES)
print('Output shape:', tuple(example_logits.shape))
print('Trainable parameters:', parameter_count)

## 3. Train a CNN with PyTorch

### 3.1 Training workflow

An **epoch** is one pass through the training examples. Each batch produces a loss, gradients and one optimiser update. Validation runs after the epoch without updating the weights.

![Training flow: for each epoch, train on every batch using forward pass, loss, backward pass and optimiser update; then validate and keep the best validation checkpoint.](assets/session12/training-workflow.svg)

### 3.2 Loading Images for Training, Validation and Testing

Run this cell to prepare the shared MNIST samples. Images are normalised to −1–1, matching the practice notebook.

To train in [the practice notebook](<Session12.1 CNN Practice.ipynb>), run its existing cells first. Then copy the code cells from **3.2, 3.3 and 3.4**, in that order, into new cells at the end. Section 3.2 supplies its own imports and constants, Section 3.3 uses the existing `model`, and Section 3.4 trains it for one epoch and prints loss and accuracy. Keep `sample_data/mnist_train_small.csv` available in the working directory.

In [ ]:
import torch
import pandas as pd
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
IMAGE_SIZE = 28
NUM_CLASSES = 10
PIXEL_MAX = 255.0
NORMALISATION_MEAN = 0.5
NORMALISATION_STD = 0.5
torch.manual_seed(RANDOM_SEED)  # Reproduce training random operations.

DATA_PATH = 'sample_data/mnist_train_small.csv'
CLASSROOM_SAMPLES = 1500
BATCH_SIZE = 32

mnist_table = pd.read_csv(DATA_PATH, header=None)  # First column is the digit label.
sample_rows, _ = train_test_split(
    mnist_table, train_size=CLASSROOM_SAMPLES,  # Select the classroom subset size.
    stratify=mnist_table.iloc[:, 0], random_state=RANDOM_SEED,
)
train_rows, held_out_rows = train_test_split(
    sample_rows, test_size=1 / 3,  # Reserve one third for validation and testing.
    stratify=sample_rows.iloc[:, 0], random_state=RANDOM_SEED,
)
validation_rows, test_rows = train_test_split(
    held_out_rows, test_size=0.5,  # Divide held-out rows equally between validation and testing.
    stratify=held_out_rows.iloc[:, 0], random_state=RANDOM_SEED,
)

def make_dataset(rows):
    images = torch.tensor(rows.iloc[:, 1:].to_numpy(), dtype=torch.float32)
    images = images.reshape(-1, 1, IMAGE_SIZE, IMAGE_SIZE) / PIXEL_MAX
    images = (images - NORMALISATION_MEAN) / NORMALISATION_STD  # Match practice inputs in [-1, 1].
    labels = torch.tensor(rows.iloc[:, 0].to_numpy(), dtype=torch.long)
    return TensorDataset(images, labels)

train_dataset = make_dataset(train_rows)
validation_dataset = make_dataset(validation_rows)
test_dataset = make_dataset(test_rows)
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,  # Shuffle training batches in this process.
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)
validation_loader = DataLoader(validation_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)
print('Training / validation / test:', len(train_dataset), len(validation_dataset), len(test_dataset))

### 3.3 Loss function, optimiser and device

`nn.CrossEntropyLoss()` compares a batch of logits with integer class labels. It includes the required log-softmax calculation, so do not add softmax to the model before this loss.

`torch.optim.Adam(model.parameters(), lr=...)` creates an optimiser for the supplied parameters. The learning rate controls the scale of its updates. `model.to(device)` moves the model to the chosen device; each batch must be moved to the same device.

[Cross-entropy loss](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) · [Adam optimiser](https://docs.pytorch.org/docs/stable/generated/torch.optim.Adam.html)

In [ ]:
from torch import nn

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

LEARNING_RATE = 0.001
model = model.to(device)  # Move parameters before constructing the optimiser.
criterion = nn.CrossEntropyLoss()  # Expect logits and class indices, rather than probabilities.
simple_optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print('Training device:', device)

### 3.4 The five operations in a training step

| Operation | Purpose |
|---|---|
| `optimiser.zero_grad()` | Clears accumulated gradients before this batch |
| `logits = model(images)` | Computes class scores with the current weights |
| `loss = criterion(logits, labels)` | Measures prediction error |
| `loss.backward()` | Computes gradients for trainable parameters |
| `optimiser.step()` | Updates parameters using those gradients |

`model.train()` activates training behaviour in layers such as Dropout and BatchNorm. It does not run the training loop itself.

The helper below also accepts an optional input adapter and a mode-setting function. The simple CNN uses the defaults; Section 5 reuses the same training step with a pretrained CNN.
Run the cell to define the helper and complete one training epoch. The printed loss and accuracy summarise all training batches. Sections 3.5–3.6 then add validation and continue training.


In [ ]:
def train_one_epoch(model, loader, optimiser, loss_function, device,
                    input_adapter=None, training_mode=None):
    if training_mode is None:
        model.train()  # Activate the model's training behaviour.
    else:
        training_mode(model)

    total_loss = 0.0
    total_correct = 0
    total_examples = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if input_adapter is not None:
            images = input_adapter(images)
        optimiser.zero_grad()  # Clear gradients from the previous batch.
        logits = model(images)  # Forward pass: compute class scores.
        loss = loss_function(logits, labels)  # Measure this batch's classification error.
        loss.backward()  # Backward pass: calculate gradients.
        optimiser.step()  # Apply one parameter update.

        batch_examples = labels.size(0)
        total_loss += loss.item() * batch_examples  # Weight the loss by batch size.
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_examples
    return total_loss / total_examples, total_correct / total_examples
import time

# Train the existing model on every training batch once.
initial_epoch_start = time.perf_counter()  # Time the first training epoch.
train_loss, train_accuracy = train_one_epoch(
    model, train_loader, simple_optimiser, criterion, device,
)
initial_epoch_seconds = time.perf_counter() - initial_epoch_start
print(f'One training epoch: loss={train_loss:.3f}, accuracy={train_accuracy:.3f}')


### 3.5 Validate without updating weights

`model.eval()` switches layers to evaluation behaviour. `torch.no_grad()` separately disables gradient recording, reducing memory use during validation and inference. Use both.

Validation loss helps us select the best epoch. The test set is reserved for final evaluation.

In [ ]:
def evaluate(model, loader, loss_function, device, input_adapter=None):
    model.eval()  # Use evaluation behaviour, including fixed BatchNorm statistics.
    total_loss = 0.0
    total_correct = 0
    all_labels = []
    all_predictions = []
    with torch.no_grad():  # Evaluate without building a gradient graph.
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            if input_adapter is not None:
                images = input_adapter(images)
            logits = model(images)
            loss = loss_function(logits, labels)
            predictions = logits.argmax(dim=1)  # Select the highest-scoring class per image.
            total_loss += loss.item() * labels.size(0)
            total_correct += (predictions == labels).sum().item()
            all_labels.append(labels.cpu())
            all_predictions.append(predictions.cpu())

    count = len(loader.dataset)
    return {
        'loss': total_loss / count,
        'accuracy': total_correct / count,
        'labels': torch.cat(all_labels).numpy(),
        'predictions': torch.cat(all_predictions).numpy(),
    }

### 3.6 Train across epochs and keep the best model

`state_dict()` contains the model's parameters and persistent buffers. A deep copy preserves the best checkpoint while later epochs continue changing the model. `load_state_dict()` restores that checkpoint.
The following cell continues from the one epoch in Section 3.4 for five more epochs, validating after each and keeping the best checkpoint from those five epochs.


In [ ]:
import copy

def fit(model, train_loader, validation_loader, optimiser, loss_function,
        device, epochs, input_adapter=None, training_mode=None):
    history = []
    best_validation_loss = float('inf')
    best_state = None
    start_time = time.perf_counter()  # Measure elapsed training and validation time.
    for epoch in range(1, epochs + 1):
        train_loss, train_accuracy = train_one_epoch(
            model, train_loader, optimiser, loss_function, device,
            input_adapter=input_adapter, training_mode=training_mode,
        )
        validation = evaluate(model, validation_loader, loss_function, device, input_adapter)
        history.append({
            'epoch': epoch, 'train_loss': train_loss, 'train_accuracy': train_accuracy,
            'validation_loss': validation['loss'], 'validation_accuracy': validation['accuracy'],
        })
        if validation['loss'] < best_validation_loss:
            best_validation_loss = validation['loss']
            best_state = copy.deepcopy(model.state_dict())  # Keep independent checkpoint tensors.
        print(f"Epoch {epoch}/{epochs}: loss={train_loss:.3f}, "
              f"accuracy={train_accuracy:.3f}, validation loss={validation['loss']:.3f}, "
              f"validation accuracy={validation['accuracy']:.3f}")
    model.load_state_dict(best_state)  # Restore the epoch with the lowest validation loss.
    return pd.DataFrame(history), time.perf_counter() - start_time

SIMPLE_EPOCHS = 5
simple_history, simple_seconds = fit(
    model, train_loader, validation_loader, simple_optimiser,
    criterion, device, epochs=SIMPLE_EPOCHS,
)
simple_seconds += initial_epoch_seconds  # Include the first epoch in the comparison time.


## 4. Evaluate the Model

### 4.1 Read the learning curves

Falling training and validation loss suggests learning that generalises to held-out examples. Falling training loss alongside rising validation loss suggests overfitting. If both remain high, inspect the model and training settings.

The checkpoint restored in Section 3 is chosen using validation loss, so its score may differ from the last row of the history. Adjust settings using validation results, rather than repeatedly consulting the test set.

In [ ]:
import matplotlib.pyplot as plt

figure, axes = plt.subplots(1, 2, figsize=(11, 4))
for prefix, label in [('train', 'Training'), ('validation', 'Validation')]:
    axes[0].plot(simple_history['epoch'], simple_history[f'{prefix}_loss'], marker='o', label=label)
    axes[1].plot(simple_history['epoch'], simple_history[f'{prefix}_accuracy'], marker='o', label=label)
for axis, title, ylabel in zip(axes, ['Loss', 'Accuracy'], ['Cross-entropy loss', 'Accuracy']):
    axis.set(title=title, xlabel='Epoch', ylabel=ylabel)
    axis.legend()
    axis.grid(alpha=0.3)
figure.tight_layout()
plt.show()

### 4.2 Test accuracy and confusion matrix

Accuracy gives the proportion of correct predictions. A confusion matrix shows which true digits are confused with which predicted digits. Rows represent true labels and columns represent predictions.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

simple_test = evaluate(model, test_loader, criterion, device)
print(f"Test loss: {simple_test['loss']:.3f}")
print(f"Test accuracy: {simple_test['accuracy']:.3f}")
print(classification_report(
    simple_test['labels'], simple_test['predictions'], labels=list(range(NUM_CLASSES)),
    zero_division=0,  # Report zero for an undefined precision or recall.
))
ConfusionMatrixDisplay.from_predictions(
    simple_test['labels'], simple_test['predictions'], labels=list(range(NUM_CLASSES)),
    cmap='Blues', colorbar=False,
)
plt.title('Simple CNN: test predictions')
plt.tight_layout()
plt.show()

### 4.3 Inspect individual predictions

Inspect mistakes as well as correct predictions. `torch.softmax(logits, dim=1)` converts class scores into probabilities for display. A high probability can still accompany a wrong prediction; these scores do not guarantee calibrated confidence.

In [ ]:
import numpy as np

mistake_indices = np.flatnonzero(simple_test['labels'] != simple_test['predictions'])
DISPLAY_COUNT = 8
display_indices = mistake_indices[:DISPLAY_COUNT]
if len(display_indices) == 0:
    display_indices = np.arange(min(DISPLAY_COUNT, len(test_dataset)))

display_images = test_dataset.tensors[0][display_indices]
model.eval()
with torch.no_grad():
    probabilities = torch.softmax(model(display_images.to(device)), dim=1).cpu()
    scores, predicted_digits = probabilities.max(dim=1)

figure, axes = plt.subplots(1, len(display_indices), figsize=(2 * len(display_indices), 3), squeeze=False)
for axis, index, image, digit, score in zip(
        axes.ravel(), display_indices, display_images, predicted_digits, scores):
    axis.imshow(image.squeeze().numpy(), cmap='gray')
    axis.set_title(f'True: {simple_test["labels"][index]}\nPredicted: {digit.item()}\nScore: {score.item():.2f}')
    axis.axis('off')
figure.tight_layout()
plt.show()

### 4.4 Discussion

- Which digit pairs are most often confused?
- Do the learning curves suggest underfitting or overfitting?
- Which single training setting would you investigate next using the validation set?

## 5. Fine-tune a CNN Model: Transfer Learning

### 5.1 Reuse learned features

Transfer learning starts with weights learned on another task. We use **ResNet18 pretrained on ImageNet**, replace its classification head with a ten-class digit classifier, and train it on the same MNIST samples used above.

First, freeze the backbone and train only the new head. Then unfreeze the final convolutional block and continue training with a smaller learning rate. The second stage is **fine-tuning**.

ImageNet contains natural colour photographs, whereas MNIST contains handwritten digits. This difference between source and target tasks means transfer learning is not guaranteed to outperform our small CNN. The models also differ in size, training duration and input representation, so the final table is a classroom comparison rather than a controlled architecture benchmark.

[PyTorch transfer-learning tutorial](https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html) · [ResNet18 model and pretrained weights](https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html)

### 5.2 Load the pretrained model and replace its head

| API | Purpose |
|---|---|
| `ResNet18_Weights.DEFAULT` | Selects the recommended pretrained weight set |
| `resnet18(weights=...)` | Constructs the model and loads those weights |
| `parameter.requires_grad = False` | Stops gradient calculation for that parameter |
| `model.fc.in_features` | Reports the feature-vector size entering the original head |
| `model.fc = nn.Linear(...)` | Replaces the original head with a new trainable layer |
| `filter(...)` | Supplies only trainable parameters to the optimiser |

Replacing the head after freezing the existing parameters leaves the new head trainable. The first call downloads weights; later calls reuse the saved copy.

In [ ]:
import os
from torchvision.models import resnet18, ResNet18_Weights

WEIGHT_CACHE = '../archive/AI_SS/session12/torch-cache'
os.makedirs(WEIGHT_CACHE, exist_ok=True)
torch.hub.set_dir(WEIGHT_CACHE)  # Keep downloaded weights in the workspace cache.
torch.manual_seed(RANDOM_SEED)
pretrained_weights = ResNet18_Weights.DEFAULT
transfer_model = resnet18(weights=pretrained_weights)  # Load ImageNet feature-extractor weights.
for parameter in transfer_model.parameters():
    parameter.requires_grad = False  # Freeze all existing parameters.
head_input_features = transfer_model.fc.in_features
transfer_model.fc = nn.Linear(head_input_features, NUM_CLASSES)  # New head starts trainable.
transfer_model = transfer_model.to(device)
print('New classification head:', transfer_model.fc)
print('Trainable parameters:', sum(p.numel() for p in transfer_model.parameters() if p.requires_grad))

### 5.3 Ready-to-run input code

Run this cell before training the pretrained model. It reuses the existing loaders and their samples.

In [ ]:
pretrained_transform = pretrained_weights.transforms()

def resnet_input(images):
    # Undo the classroom normalisation before applying the pretrained transform.
    images_01 = images * NORMALISATION_STD + NORMALISATION_MEAN
    images_rgb = images_01.repeat(1, 3, 1, 1)  # Supply three colour channels.
    return pretrained_transform(images_rgb)

transfer_train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0,
    generator=torch.Generator().manual_seed(RANDOM_SEED),
)

### 5.4 Train the new classification head

Freezing parameters and choosing the model's mode are separate actions. BatchNorm running statistics can change in training mode even when its parameters are frozen. Keep the frozen backbone in evaluation mode and place only the new head in training mode.

The training loop still computes loss and calls `backward()` and `step()`, but the optimiser updates only the new head.

In [ ]:
def head_training_mode(model):
    model.eval()  # Preserve the frozen backbone's BatchNorm running statistics.
    model.fc.train()  # Activate training behaviour only in the new head.

HEAD_LEARNING_RATE = 0.001
HEAD_EPOCHS = 2
head_optimiser = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, transfer_model.parameters()),
    lr=HEAD_LEARNING_RATE,  # Learning rate for the randomly initialised head.
)
head_history, head_seconds = fit(
    transfer_model, transfer_train_loader, validation_loader, head_optimiser,
    criterion, device, epochs=HEAD_EPOCHS,
    input_adapter=resnet_input, training_mode=head_training_mode,
)
head_state = copy.deepcopy(transfer_model.state_dict())  # Preserve the best head-only checkpoint.
head_validation = evaluate(transfer_model, validation_loader, criterion, device, resnet_input)

### 5.5 Unfreeze the final block and fine-tune

`layer4` is the final convolutional block in ResNet18. Set its parameters' `requires_grad` to `True`, then construct a new optimiser that includes those parameters as well as the head. Simply unfreezing a layer does not add its parameters to an existing optimiser.

Use a smaller learning rate to adjust pretrained features gradually. Keep the earlier frozen blocks in evaluation mode, while `layer4` and the head use training mode. This permits the final block's BatchNorm statistics to adapt.

In [ ]:
for parameter in transfer_model.layer4.parameters():
    parameter.requires_grad = True  # Allow gradients for the final convolutional block.

def fine_tuning_mode(model):
    model.eval()  # Keep earlier frozen blocks and their running statistics fixed.
    model.layer4.train()  # Adapt the final block, including its BatchNorm statistics.
    model.fc.train()

FINE_TUNE_LEARNING_RATE = 0.0001
FINE_TUNE_EPOCHS = 2
fine_tune_optimiser = torch.optim.Adam(
    filter(lambda parameter: parameter.requires_grad, transfer_model.parameters()),
    lr=FINE_TUNE_LEARNING_RATE,  # Smaller updates for pretrained parameters.
)
fine_tune_history, fine_tune_seconds = fit(
    transfer_model, transfer_train_loader, validation_loader, fine_tune_optimiser,
    criterion, device, epochs=FINE_TUNE_EPOCHS,
    input_adapter=resnet_input, training_mode=fine_tuning_mode,
)
fine_tune_validation = evaluate(transfer_model, validation_loader, criterion, device, resnet_input)
fine_tune_state = copy.deepcopy(transfer_model.state_dict())
print('Head-only validation loss:', round(head_validation['loss'], 3))
print('Fine-tuned validation loss:', round(fine_tune_validation['loss'], 3))

### 5.6 Compare the trained models

Evaluate both saved transfer-learning checkpoints on the same test samples. Select a checkpoint for future use by **validation loss**, not by test accuracy. Fine-tuning can make validation performance worse, so retain the head-only checkpoint as a candidate.

The fine-tuned model's training time includes both transfer-learning stages. It excludes the weight download, final testing and model construction. CUDA and MPS operations may be asynchronous; these elapsed times are approximate classroom measurements.

In [ ]:
transfer_model.load_state_dict(head_state)
head_test = evaluate(transfer_model, test_loader, criterion, device, resnet_input)
transfer_model.load_state_dict(fine_tune_state)
fine_tune_test = evaluate(transfer_model, test_loader, criterion, device, resnet_input)

comparison = pd.DataFrame([
    {'Model': 'Simple CNN from scratch', 'Test accuracy': simple_test['accuracy'],
     'Test loss': simple_test['loss'], 'Training seconds': simple_seconds},
    {'Model': 'ResNet18: head only', 'Test accuracy': head_test['accuracy'],
     'Test loss': head_test['loss'], 'Training seconds': head_seconds},
    {'Model': 'ResNet18: fine-tuned', 'Test accuracy': fine_tune_test['accuracy'],
     'Test loss': fine_tune_test['loss'], 'Training seconds': head_seconds + fine_tune_seconds},
])
print(comparison.round(3).to_string(index=False))

selected_transfer_state = (
    fine_tune_state if fine_tune_validation['loss'] < head_validation['loss'] else head_state
)
transfer_model.load_state_dict(selected_transfer_state)  # Select using validation loss only.
transfer_model.eval()